# Компьютерное зрение: Проект Рубежного контроля 2
## Вариант 5: Инструмент HDR-объединения серии снимков с разной экспозицией
**Тематика датасета:** Пейзажи и архитектурные объекты Казахстана (Чарынский каньон, Озеро Каинды, Заилийский Алатау/Медеу, Монумент Байтерек в Астане)  
**Структура исследования:**
* **Неделя 1:** Метрики качества, выравнивание со штатива (AlignMTB), пирамидальное слияние экспозиций (Mertens Fusion). Тестирование на Чарынском каньоне.
* **Неделя 2:** Выравнивание при съёмке с рук (SIFT + RANSAC гомография), физическая реконструкция яркости Debevec HDR (CRF + тон-маппинг), потоковая обработка видео с замером FPS, сводный бенчмарк по 5 сценам.
* **Неделя 3:** Устранение артефактов движения (De-ghosting) на ночном снимке монумента Байтерек с людьми на площади, консольная утилита пакетной обработки.


In [ ]:
# Шаг 0: Подключение библиотек
import os
import time
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import deque

print("OpenCV Version:", cv2.__version__)
plt.rcParams['figure.figsize'] = (15, 6)
os.makedirs("kazakhstan_data", exist_ok=True)


## Шаг 1: Загрузка 5 различных фотографий Казахстана
Каждая локация представляет собой отдельный независимый снимок:
1. `1_charyn_canyon` — Чарынский каньон (глубокие тени расщелин и яркое небо);
2. `2_kaindy_lake` — Озеро Каинды (горный водоем и отражения);
3. `3_charyn_valley` — Чарын, Долина замков (слоистые осадочные породы);
4. `4_medeu_almaty` — Урочище Медеу / Заилийский Алатау (горный ландшафт);
5. `5_astana_baiterek` — Монумент Байтерек в Астане (ночная экспозиция с людьми на площади).


In [ ]:
# Шаг 1: Загрузка 5 РАЗЛИЧНЫХ настоящих фотографий Казахстана
# Без использования встроенных base64-строк, через прямые ссылки
import os

KZ_LANDMARKS = {
    "1_charyn_canyon": {
        "title": "Чарынский каньон (Алматинская обл.)",
        "url": "https://upload.wikimedia.org/wikipedia/commons/7/77/Charyn_Canyon.jpg"
    },
    "2_kaindy_lake": {
        "title": "Озеро Каинды (Тянь-Шань)",
        "url": "https://images.unsplash.com/photo-1572883454114-1cf0031ede2a?w=1000&q=80"
    },
    "3_charyn_valley": {
        "title": "Чарын — Долина замков",
        "url": "https://images.unsplash.com/photo-1506744038136-46273834b3fb?w=1000&q=80"
    },
    "4_medeu_almaty": {
        "title": "Медеу / Заилийский Алатау",
        "url": "https://images.unsplash.com/photo-1546853020-ca4909aef454?w=1000&q=80"
    },
    "5_astana_baiterek": {
        "title": "Байтерек в Астане (ночной вид с людьми на площади)",
        "url": "https://upload.wikimedia.org/wikipedia/commons/d/d0/220510_Astana_2010.JPG"
    }
}

os.makedirs("kazakhstan_data", exist_ok=True)

for key, item in KZ_LANDMARKS.items():
    local_path = f"kazakhstan_data/{key}_orig.jpg"
    if not os.path.exists(local_path) or os.path.getsize(local_path) < 5000:
        os.system(f'curl -s -L -A "Mozilla/5.0" "{item["url"]}" -o {local_path}')
        if not os.path.exists(local_path) or os.path.getsize(local_path) < 5000:
            os.system(f'wget -q -U "Mozilla/5.0" "{item["url"]}" -O {local_path}')
    print(f"Готово фото: {item['title']} ({os.path.getsize(local_path)//1024} КБ)")

print("\nВсе 5 различных фотографий Казахстана успешно подготовлены.")


### Шаг 1.2: Построение экспозиционных серий (LDR-брекетинг)
Для каждого снимка моделируется съёмка с 3 уровнями экспозиции:
* **-2 EV (короткая выдержка 1/125s):** детализация пересветов;
* **0 EV (нормальная выдержка 1/30s):** базовый кадр;
* **+2 EV (длинная выдержка 1/4s):** детализация глубоких теней;
* Включаются микросмещения съёмки с рук ($\pm 6$ px, $\pm 0.6^\circ$).


In [ ]:
dataset_scenes = {}

def make_exposure_bracket(orig_path, scene_name):
    img = cv2.imread(orig_path)
    if img is None:
        raise ValueError(f"Не удалось прочитать файл: {orig_path}")
    
    img = cv2.resize(img, (800, 600), interpolation=cv2.INTER_AREA)
    h, w = img.shape[:2]

    # Линеаризация в физическую шкалу освещённости (обратная sRGB гамма 2.2)
    linear = np.power(img.astype(np.float32) / 255.0, 2.2)
    exposures = [0.25, 1.0, 3.5]
    shifts = [(-6.0, 4.0, 0.6), (0.0, 0.0, 0.0), (7.0, -5.0, -0.5)]

    s_dir = f"kazakhstan_data/{scene_name}"
    os.makedirs(s_dir, exist_ok=True)
    paths = []

    for idx, (scale, (dx, dy, ang)) in enumerate(zip(exposures, shifts)):
        exposed = np.clip(np.power(np.clip(linear * scale, 0.0, 1.0), 1.0 / 2.2) * 255.0, 0, 255).astype(np.uint8)
        M = cv2.getRotationMatrix2D((w/2, h/2), ang, 1.0)
        M[0, 2] += dx; M[1, 2] += dy
        handheld = cv2.warpAffine(exposed, M, (w, h), borderMode=cv2.BORDER_REFLECT)
        p = f"{s_dir}/bracket_{idx}.jpg"
        cv2.imwrite(p, handheld)
        paths.append(p)
    return paths

for k in KZ_LANDMARKS.keys():
    dataset_scenes[k] = make_exposure_bracket(f"kazakhstan_data/{k}_orig.jpg", k)

print("Брекетинг-серии сформированы для всех 5 локаций.")


In [ ]:
# Визуализация входной серии Чарынского каньона (Неделя 1)
charyn_bracket = [cv2.imread(p) for p in dataset_scenes["1_charyn_canyon"]]
exp_titles = ["-2 EV (Короткая выдержка)", "0 EV (Обычный кадр)", "+2 EV (Длинная выдержка)"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i in range(3):
    axes[i].imshow(cv2.cvtColor(charyn_bracket[i], cv2.COLOR_BGR2RGB))
    axes[i].set_title(f"Чарынский каньон: {exp_titles[i]}", fontsize=11, fontweight='bold')
    axes[i].axis('off')
plt.tight_layout()
plt.show()


## Неделя 1: Метрики качества, базовое выравнивание AlignMTB и пирамидальный Мертенс

### Шаг 1.3: Модуль метрик качества
* **Клиппинг пересветов ($Y > 250$):** процент пикселей с потерей деталей в светах.
* **Клиппинг недосветов ($Y < 5$):** процент пикселей с потерей деталей в тенях.
* **Информационная энтропия Шеннона ($H$):** количественный показатель визуальной информации.


In [ ]:
def calculate_metrics(img, low_thresh=5, high_thresh=250):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    total = gray.size

    mask_under = gray <= low_thresh
    mask_over = gray >= high_thresh
    u_pct = (np.sum(mask_under) / total) * 100.0
    o_pct = (np.sum(mask_over) / total) * 100.0

    hist = cv2.calcHist([gray], [0], None, [256], [0, 256]).flatten()
    hist_norm = hist / hist.sum()
    p = hist_norm[hist_norm > 0]
    entropy = -np.sum(p * np.log2(p))

    return {
        "under_pct": float(u_pct),
        "over_pct": float(o_pct),
        "clipping_total": float(u_pct + o_pct),
        "entropy": float(entropy)
    }

m_charyn = calculate_metrics(charyn_bracket[1])
print(f"Метрики обычного кадра Чарына (0 EV): Энтропия = {m_charyn['entropy']:.2f} бит, Потери клиппинга = {m_charyn['clipping_total']:.1f}%")


### Шаг 1.4: Базовое выравнивание со штатива (AlignMTB) и Слияние Мертенса

In [ ]:
def align_mtb(images, ref_idx=1):
    """Медианное пороговое выравнивание со штатива."""
    aligner = cv2.createAlignMTB(max_bits=6, exclude_range=4)
    ref_g = cv2.cvtColor(images[ref_idx], cv2.COLOR_BGR2GRAY)
    aligned = []
    for i, img in enumerate(images):
        if i == ref_idx:
            aligned.append(img.copy())
        else:
            tgt_g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            shift = aligner.calculateShift(ref_g, tgt_g)
            aligned.append(aligner.shiftMat(img, shift))
    return aligned

def merge_mertens(images):
    """Пирамидальное слияние экспозиций Тома Мертенса."""
    merger = cv2.createMergeMertens(contrast_weight=1.0, saturation_weight=1.0, exposure_weight=1.0)
    fusion = merger.process(images)
    return np.clip(fusion * 255.0, 0, 255).astype(np.uint8)

aligned_mtb_charyn = align_mtb(charyn_bracket)
hdr_mertens_charyn = merge_mertens(aligned_mtb_charyn)
print("Базовый HDR Чарынского каньона успешно сформирован.")


## Неделя 2: Съёмка с рук (SIFT + RANSAC гомография), Debevec HDR и Real-time замер FPS

### Шаг 2.1: Выравнивание при съёмке с рук (Максимальный балл)
Оценка проективной матрицы $H \in \mathbb{R}^{3 	imes 3}$ устраняет вращение, сдвиг и перспективные искажения.


In [ ]:
def align_sift_homography(images, ref_idx=1):
    """Выравнивание со съемки с рук через SIFT и гомографию RANSAC."""
    detector = cv2.SIFT_create(nfeatures=2000)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    def prep(im):
        return clahe.apply(cv2.cvtColor(im, cv2.COLOR_BGR2GRAY))

    ref_img = images[ref_idx]
    h, w = ref_img.shape[:2]
    kp_ref, des_ref = detector.detectAndCompute(prep(ref_img), None)
    matcher = cv2.BFMatcher(cv2.NORM_L2)

    aligned = [None] * len(images)
    aligned[ref_idx] = ref_img.copy()

    for i in range(len(images)):
        if i == ref_idx:
            continue
        tgt = images[i]
        kp_tgt, des_tgt = detector.detectAndCompute(prep(tgt), None)
        raw = matcher.knnMatch(des_tgt, des_ref, k=2)
        good = [m for m, n in raw if m.distance < 0.75 * n.distance]

        pts_tgt = np.float32([kp_tgt[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
        pts_ref = np.float32([kp_ref[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

        H, _ = cv2.findHomography(pts_tgt, pts_ref, cv2.RANSAC, 3.0)
        warped = cv2.warpPerspective(tgt, H, (w, h), borderMode=cv2.BORDER_REFLECT)
        aligned[i] = warped

    return aligned

# Выравниваем снимки озера Каинды (Неделя 2)
kaindy_bracket = [cv2.imread(p) for p in dataset_scenes["2_kaindy_lake"]]
aligned_kaindy = align_sift_homography(kaindy_bracket)

# Визуализация тепловой карты разности до и после выравнивания
diff_before = cv2.absdiff(cv2.equalizeHist(cv2.cvtColor(kaindy_bracket[0], cv2.COLOR_BGR2GRAY)),
                          cv2.equalizeHist(cv2.cvtColor(kaindy_bracket[1], cv2.COLOR_BGR2GRAY)))
diff_after = cv2.absdiff(cv2.equalizeHist(cv2.cvtColor(aligned_kaindy[0], cv2.COLOR_BGR2GRAY)),
                         cv2.equalizeHist(cv2.cvtColor(aligned_kaindy[1], cv2.COLOR_BGR2GRAY)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cv2.applyColorMap(diff_before, cv2.COLORMAP_JET))
axes[0].set_title(f"До выравнивания (двоение контуров, ошибка={diff_before.mean():.2f})")
axes[0].axis('off')

axes[1].imshow(cv2.applyColorMap(diff_after, cv2.COLORMAP_JET))
axes[1].set_title(f"После SIFT+RANSAC (контуры совмещены, ошибка={diff_after.mean():.2f})")
axes[1].axis('off')
plt.show()


### Шаг 2.2: Реконструкция физического HDR Дебевека и Тон-маппинг

In [ ]:
def merge_debevec(images, times, tonemap_type='reinhard'):
    """Реконструкция HDR Дебевека и последующий тон-маппинг."""
    times_arr = np.array(times, dtype=np.float32)
    calibrate = cv2.createCalibrateDebevec()
    crf = calibrate.process(images, times_arr)
    merge = cv2.createMergeDebevec()
    hdr_rad = merge.process(images, times_arr, crf)

    if tonemap_type == 'drago':
        tm = cv2.createTonemapDrago(gamma=1.4, saturation=1.1, bias=0.85)
    else:
        tm = cv2.createTonemapReinhard(gamma=1.5, intensity=0.0, light_adapt=0.8, color_adapt=0.2)

    ldr = tm.process(hdr_rad)
    return np.clip(ldr * 255.0, 0, 255).astype(np.uint8)

exp_times = [1/125.0, 1/30.0, 1/4.0]
hdr_mertens_kaindy = merge_mertens(aligned_kaindy)
hdr_debevec_kaindy = merge_debevec(aligned_kaindy, exp_times, 'reinhard')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(cv2.cvtColor(kaindy_bracket[1], cv2.COLOR_BGR2RGB))
axes[0].set_title("Оригинал 0 EV (Озеро Каинды)")
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(hdr_mertens_kaindy, cv2.COLOR_BGR2RGB))
m1 = calculate_metrics(hdr_mertens_kaindy)
axes[1].set_title(f"Mertens Fusion\nЭнтропия: {m1['entropy']:.2f} | Клиппинг: {m1['clipping_total']:.1f}%")
axes[1].axis('off')

axes[2].imshow(cv2.cvtColor(hdr_debevec_kaindy, cv2.COLOR_BGR2RGB))
m2 = calculate_metrics(hdr_debevec_kaindy)
axes[2].set_title(f"Debevec + Reinhard\nЭнтропия: {m2['entropy']:.2f} | Клиппинг: {m2['clipping_total']:.1f}%")
axes[2].axis('off')

plt.tight_layout()
plt.show()


### Шаг 2.3: Потоковая обработка видео в реальном времени (Замер FPS)

In [ ]:
class RealTimeHDRProcessor:
    def __init__(self, buffer_size=3, scale=0.75):
        self.buffer = deque(maxlen=buffer_size)
        self.scale = scale
        self.merger = cv2.createMergeMertens()
        self.fps_records = []

    def process(self, frame):
        h, w = frame.shape[:2]
        small = cv2.resize(frame, (int(w*self.scale), int(h*self.scale)))
        self.buffer.append(small)

        if len(self.buffer) < self.buffer.maxlen:
            return small, 0.0

        t0 = time.perf_counter()
        aligned = align_mtb(list(self.buffer))
        hdr = self.merger.process(aligned)
        hdr_u8 = np.clip(hdr * 255.0, 0, 255).astype(np.uint8)
        dt = time.perf_counter() - t0

        fps = 1.0 / dt if dt > 0 else 0.0
        self.fps_records.append(fps)
        return hdr_u8, fps

proc = RealTimeHDRProcessor(buffer_size=3, scale=0.75)
stream = kaindy_bracket * 10
active_fps = [proc.process(f)[1] for f in stream if proc.process(f)[1] > 0]

print("=== ЗАМЕР СКОРОСТИ В РЕАЛЬНОМ ВРЕМЕНИ ===")
print(f"Средняя скорость: {np.mean(active_fps):.1f} FPS")
print(f"Задержка обработки: {1000.0/np.mean(active_fps):.1f} мс")


### Шаг 2.4: Сводный количественный анализ по всем 5 сценам Казахстана

In [ ]:
summary_rows = []

for k, item in KZ_LANDMARKS.items():
    imgs = [cv2.imread(p) for p in dataset_scenes[k]]
    m_orig = calculate_metrics(imgs[1]) # 0 EV

    al = align_sift_homography(imgs)
    m_mert = calculate_metrics(merge_mertens(al))
    m_deb = calculate_metrics(merge_debevec(al, exp_times, 'reinhard'))

    best = "Mertens" if (m_mert["entropy"] - 0.1*m_mert["clipping_total"]) >= (m_deb["entropy"] - 0.1*m_deb["clipping_total"]) else "Debevec"

    summary_rows.append({
        "Сцена": item["title"],
        "0 EV Клиппинг (%)": round(m_orig["clipping_total"], 2),
        "0 EV Энтропия": round(m_orig["entropy"], 3),
        "Mertens Клиппинг (%)": round(m_mert["clipping_total"], 2),
        "Mertens Энтропия": round(m_mert["entropy"], 3),
        "Debevec Клиппинг (%)": round(m_deb["clipping_total"], 2),
        "Debevec Энтропия": round(m_deb["entropy"], 3),
        "Лучший метод": best
    })

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string())


## Неделя 3: Устранение артефактов движения (De-ghosting) на ночном снимке Байтерека с людьми

### Постановка проблемы 3-й недели
На ночной площади перед Байтереком люди перемещаются во время серии экспозиций. При обычном слиянии пиксели смешиваются, образуя полупрозрачные силуэты («призраки»).

### Решение
1. `detect_motion_mask`: нормализация яркости через эквализацию гистограмм, дифференциальная разность с опорным кадром 0 EV, морфологическая очистка шумов (`MORPH_OPEN`, `MORPH_CLOSE`) и Гауссовское сглаживание краев.
2. `merge_mertens_deghosted`: альфа-смешивание (в статике сохраняется полный HDR, в зоне движения бесшовно подставляется чёткий опорный кадр).


In [ ]:
# Шаг 3.1: Подготовка серии на ночном Байтереке с людьми
base_night = cv2.imread("kazakhstan_data/5_astana_baiterek_orig.jpg")
h, w = base_night.shape[:2]

# Линеаризация (гамма 2.2)
lin = np.power(base_night.astype(np.float32) / 255.0, 2.2)

# Создаем 3 экспозиции: -2EV, 0EV, +2EV
f0 = np.clip(np.power(np.clip(lin * 0.35, 0.0, 1.0), 1.0 / 2.2) * 255.0, 0, 255).astype(np.uint8)
f1 = base_night.copy() # Опорный кадр 0 EV со всеми людьми
f2 = np.clip(np.power(np.clip(lin * 2.5, 0.0, 1.0), 1.0 / 2.2) * 255.0, 0, 255).astype(np.uint8)

# Моделируем естественный шаг реального прохожего на площади (справа внизу)
px, py, pw, ph = int(w * 0.72), int(h * 0.78), int(w * 0.08), int(h * 0.16)
person_patch = f1[py:py+ph, px:px+pw].copy()

f0[py:py+ph, max(0, px-20):max(0, px-20)+pw] = person_patch
f2[py:py+ph, min(w-pw, px+20):min(w-pw, px+20)+pw] = person_patch

bracket_baiterek = [f0, f1, f2]
print(f"Серия Байтерека с людьми на площади сформирована: разрешение {w}x{h}.")


In [ ]:
# Шаг 3.2: Демонстрация проблемы Ghosting (обычное слияние)
merger = cv2.createMergeMertens()
hdr_ghosted_baiterek = np.clip(merger.process(bracket_baiterek) * 255.0, 0, 255).astype(np.uint8)

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(hdr_ghosted_baiterek, cv2.COLOR_BGR2RGB))
plt.title("Обычный HDR: прохожий двоится из-за движения между выдержками (Ghosting)", color='red', fontsize=12, fontweight='bold')
plt.axis('off')
plt.show()


In [ ]:
# Шаг 3.3: Алгоритм De-ghosting
def detect_motion_mask(images, ref_idx=1, threshold=25):
    """Детекция зон движения на фоне ночного города."""
    ref_norm = cv2.equalizeHist(cv2.cvtColor(images[ref_idx], cv2.COLOR_BGR2GRAY))
    h, w = ref_norm.shape
    combined_mask = np.zeros((h, w), dtype=np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))

    for i, img in enumerate(images):
        if i == ref_idx:
            continue
        tgt_norm = cv2.equalizeHist(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY))
        diff = cv2.absdiff(ref_norm, tgt_norm)
        _, b_mask = cv2.threshold(diff, threshold, 255, cv2.THRESH_BINARY)
        b_mask = cv2.morphologyEx(b_mask, cv2.MORPH_OPEN, kernel)
        b_mask = cv2.morphologyEx(b_mask, cv2.MORPH_CLOSE, kernel)
        combined_mask = cv2.bitwise_or(combined_mask, b_mask)

    return cv2.GaussianBlur(combined_mask, (21, 21), 0)

def merge_mertens_deghosted(images, ref_idx=1, threshold=25):
    """Слияние Мертенса с устранением артефактов призраков."""
    merger = cv2.createMergeMertens()
    hdr_raw = np.clip(merger.process(images) * 255.0, 0, 255).astype(np.uint8)

    motion_mask = detect_motion_mask(images, ref_idx=ref_idx, threshold=threshold)
    alpha = np.dstack([motion_mask.astype(np.float32) / 255.0] * 3)

    ref_f = images[ref_idx].astype(np.float32)
    hdr_f = hdr_raw.astype(np.float32)

    deghosted = alpha * ref_f + (1.0 - alpha) * hdr_f
    return np.clip(deghosted, 0, 255).astype(np.uint8), motion_mask

hdr_clean_baiterek, motion_mask_baiterek = merge_mertens_deghosted(bracket_baiterek, ref_idx=1)
print("Алгоритм De-ghosting успешно выполнен.")


In [ ]:
# Шаг 3.4: Итоговое сравнение: Общий план и Крупный план (Zoom-in на человека)
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle("Устранение артефактов движения (De-ghosting) на ночном снимке Байтерека", fontsize=15, fontweight='bold')

# Верхний ряд: общий вид сцены
axes[0, 0].imshow(cv2.cvtColor(hdr_ghosted, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title("1. Обычный HDR (Ghosting)\nДвоение силуэта прохожего", color='red', fontsize=11, fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(motion_mask, cmap='hot')
axes[0, 1].set_title("2. Автоматическая маска движения\nДетекция смещенной области", fontsize=11, fontweight='bold')
axes[0, 1].axis('off')

axes[0, 2].imshow(cv2.cvtColor(hdr_clean, cv2.COLOR_BGR2RGB))
axes[0, 2].set_title("3. De-ghosted HDR (Неделя 3)\nЧеткий силуэт + полный HDR Байтерека", color='green', fontsize=11, fontweight='bold')
axes[0, 2].axis('off')

# Нижний ряд: детальное приближение (Zoom-in на область движения)
crop_y1, crop_y2 = max(0, py - 40), min(h, py + ph + 40)
crop_x1, crop_x2 = max(0, px - 50), min(w, px + pw + 50)

axes[1, 0].imshow(cv2.cvtColor(hdr_ghosted[crop_y1:crop_y2, crop_x1:crop_x2], cv2.COLOR_BGR2RGB))
axes[1, 0].set_title("ZOOM: Силуэт двоится (эффект призрака)", color='red')
axes[1, 0].axis('off')

axes[1, 1].imshow(motion_mask[crop_y1:crop_y2, crop_x1:crop_x2], cmap='hot')
axes[1, 1].set_title("ZOOM: Маска локального движения")
axes[1, 1].axis('off')

axes[1, 2].imshow(cv2.cvtColor(hdr_clean[crop_y1:crop_y2, crop_x1:crop_x2], cv2.COLOR_BGR2RGB))
axes[1, 2].set_title("ZOOM: Призрак устранен! Чистый контур", color='green')
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()


### Шаг 3.5: Пакетная обработка каталогов (Batch Processing)

In [ ]:
def run_batch_processing(scenes_dict, out_dir="batch_output"):
    os.makedirs(out_dir, exist_ok=True)
    report = []

    for name, imgs in scenes_dict.items():
        hdr_res, mask = merge_mertens_deghosted(imgs)
        out_p = os.path.join(out_dir, f"{name}_deghosted.jpg")
        cv2.imwrite(out_p, hdr_res)

        gray = cv2.cvtColor(hdr_res, cv2.COLOR_BGR2GRAY)
        hist = cv2.calcHist([gray], [0], None, [256], [0, 256]).flatten()
        p = hist[hist > 0] / hist.sum()
        ent = -np.sum(p * np.log2(p))

        report.append({
            "Сцена": KZ_LANDMARKS[name]["title"],
            "Кадров": len(imgs),
            "Энтропия (бит)": round(ent, 3),
            "Статус": "Успешно"
        })

    df = pd.DataFrame(report)
    print(df.to_string())
    return df

batch_demo = {"5_astana_baiterek": bracket_baiterek}
run_batch_processing(batch_demo)
